# Question Answering with LangChain, OpenAI, and MultiQuery Retriever

This interactive workbook demonstrates example of Elasticsearch's [MultiQuery Retriever](https://api.python.langchain.com/en/latest/retrievers/langchain.retrievers.multi_query.MultiQueryRetriever.html) to generate similar queries for a given user input and apply all queries to retrieve a larger set of relevant documents from a vectorstore.

Before we begin, we first split the fictional workplace documents into passages with `langchain` and uses OpenAI to transform these passages into embeddings and then store these into Elasticsearch.

We will then ask a question, generate similar questions using langchain and OpenAI, retrieve relevant passages from the vector store, and use langchain and OpenAI again to provide a summary for the questions.

## Install packages and import modules

In [1]:
!python -m pip install -qU jq lark langchain langchain-elasticsearch langchain_openai tiktoken

from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_elasticsearch import ElasticsearchStore
from langchain_openai.llms import OpenAI
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from getpass import getpass


## Connect to Elasticsearch

ℹ️ We're using an Elastic Cloud deployment of Elasticsearch for this notebook. If you don't have an Elastic Cloud deployment, sign up [here](https://cloud.elastic.co/registration?utm_source=github&utm_content=elasticsearch-labs-notebook) for a free trial. 

We'll use the **Cloud ID** to identify our deployment, because we are using Elastic Cloud deployment. To find the Cloud ID for your deployment, go to https://cloud.elastic.co/deployments and select your deployment.

We will use [ElasticsearchStore](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html) to connect to our elastic cloud deployment, This would help create and index data easily.  We would also send list of documents that we created in the previous step

In [2]:
import os
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_elasticsearch import ElasticsearchStore

load_dotenv()

embeddings = OpenAIEmbeddings(
    api_key=os.getenv("OPENAI_API_KEY")
)

vectorstore = ElasticsearchStore(
    es_cloud_id=os.getenv("ELASTIC_CLOUD_ID"),
    es_api_key=os.getenv("ELASTIC_API_KEY"),
    index_name="workplace_docs",
    embedding=embeddings,
)

print("✅ ElasticsearchStore conectado")

✅ ElasticsearchStore conectado


## Indexing Data into Elasticsearch
Let's download the sample dataset and deserialize the document.

In [3]:
from urllib.request import urlopen
import json

url = "https://raw.githubusercontent.com/elastic/elasticsearch-labs/main/example-apps/chatbot-rag-app/data/data.json"

response = urlopen(url)
data = json.load(response)

with open("temp.json", "w") as json_file:
    json.dump(data, json_file)

### Split Documents into Passages

We’ll chunk documents into passages in order to improve the retrieval specificity and to ensure that we can provide multiple passages within the context window of the final question answering prompt.

Here we are chunking documents into 800 token passages with an overlap of 400 tokens.

Here we are using a simple splitter but Langchain offers more advanced splitters to reduce the chance of context being lost.

In [4]:
from langchain_community.document_loaders import JSONLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


def metadata_func(record: dict, metadata: dict) -> dict:
    metadata["name"] = record.get("name")
    metadata["summary"] = record.get("summary")
    metadata["url"] = record.get("url")
    metadata["category"] = record.get("category")
    metadata["updated_at"] = record.get("updated_at")
    return metadata


loader = JSONLoader(
    file_path="temp.json",
    jq_schema=".[]",
    content_key="content",
    metadata_func=metadata_func,
)

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=800,
    chunk_overlap=400,
)

docs = loader.load_and_split(text_splitter=text_splitter)

C:\Users\con2m\AppData\Local\Temp\ipykernel_8856\1964889034.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import JSONLoader


### Bulk Import Passages

Now that we have split each document into the chunk size of 800, we will now index data to elasticsearch using [ElasticsearchStore.from_documents](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html#langchain.vectorstores.elasticsearch.ElasticsearchStore.from_documents).

We will use Cloud ID, Password and Index name values set in the `Create cloud deployment` step.

In [8]:
import os
from dotenv import load_dotenv

load_dotenv()

ELASTIC_CLOUD_ID = os.getenv("ELASTIC_CLOUD_ID")
ELASTIC_API_KEY = os.getenv("ELASTIC_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

print("Cloud:", ELASTIC_CLOUD_ID is not None)
print("Elastic:", ELASTIC_API_KEY is not None)
print("OpenAI:", OPENAI_API_KEY is not None)

Cloud: True
Elastic: True
OpenAI: True


In [9]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_elasticsearch import ElasticsearchStore

# Crear el índice e indexar los documentos
vectorstore = ElasticsearchStore.from_documents(
    documents=docs,
    embedding=embeddings,
    index_name="workplace_docs",
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
)

# LLM
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    api_key=OPENAI_API_KEY,
)

# Retriever
retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 4
    }
)

# Question Answering with MultiQuery Retriever

Now that we have the passages stored in Elasticsearch, we can now ask a question to get the relevant passages.

In [11]:
import langchain_core.documents as docs

print(dir(docs))

['BaseDocumentCompressor', 'BaseDocumentTransformer', 'Document']


In [12]:
import langchain_core.prompts as prompts
print("Prompts OK")

Prompts OK


In [14]:
import logging

from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

logging.basicConfig()

LLM_CONTEXT_PROMPT = ChatPromptTemplate.from_template(
    """
You are an assistant for question-answering tasks.

Use the following pieces of retrieved context to answer the question.

If you don't know the answer, say you don't know.

Context:
{context}

Question:
{question}

Answer:
"""
)

LLM_DOCUMENT_PROMPT = PromptTemplate.from_template(
    """
---
SOURCE: {name}

{page_content}
---
"""
)


def combine_documents(docs):
    formatted = []

    for doc in docs:
        formatted.append(
            LLM_DOCUMENT_PROMPT.format(
                name=doc.metadata.get("name", "Unknown"),
                page_content=doc.page_content,
            )
        )

    return "\n\n".join(formatted)


context = RunnableParallel(
    context=retriever | combine_documents,
    question=RunnablePassthrough(),
)

chain = context | LLM_CONTEXT_PROMPT | llm

response = chain.invoke("what is the nasa sales team?")

print(response.content)

The NASA sales team refers to the North America South America region within the sales organization. This region includes the United States, Canada, Mexico, as well as Central and South America. The team is led by two Area Vice-Presidents: Laura Martinez is the Area Vice-President of North America, and Gary Johnson is the Area Vice-President of South America. The sales team in this region is responsible for identifying and pursuing new business opportunities, nurturing existing client relationships, and ensuring customer satisfaction.


**Generate at least two new iteratioins of the previous cells - Be creative.** Did you master Multi-
Query Retriever concepts through this lab?

# Lab Summary: Retrieval-Augmented Generation (RAG) with Elasticsearch and LangChain

In this lab, I built a Retrieval-Augmented Generation (RAG) pipeline using Elasticsearch as the vector database and OpenAI embeddings for semantic search.

The workflow started by connecting securely to an Elasticsearch Cloud deployment using a Cloud ID and an API Key stored in environment variables. After configuring the connection, I downloaded a JSON dataset containing company knowledge documents.

The dataset was loaded with a JSON loader and split into smaller passages using a recursive text splitter. Each document was divided into chunks of approximately 800 tokens with an overlap of 400 tokens to preserve context while improving retrieval accuracy.

Next, the document chunks were converted into vector embeddings using OpenAI's embedding model and indexed into an Elasticsearch vector index. This enabled semantic similarity search over the document collection.

A retriever was then created from the Elasticsearch vector store to perform semantic searches based on user questions. The retrieved passages were formatted into a context prompt and combined with a ChatOpenAI language model to generate grounded answers.

Finally, a complete RAG pipeline was built using LangChain runnables. When a user submits a question, the retriever searches for the most relevant passages, the retrieved context is injected into a prompt, and the language model produces a context-aware response based only on the indexed documents.

## Technologies Used

- Elasticsearch Cloud
- LangChain
- LangChain Elasticsearch
- OpenAI Embeddings
- ChatOpenAI (GPT-4o-mini)
- Recursive Character Text Splitter
- JSONLoader
- RunnableParallel
- Prompt Templates

## Key Concepts Learned

- Connecting to Elasticsearch Cloud using API Keys.
- Loading structured JSON documents.
- Chunking documents for semantic retrieval.
- Creating vector embeddings with OpenAI.
- Indexing documents into Elasticsearch.
- Performing semantic similarity search.
- Building a Retrieval-Augmented Generation (RAG) pipeline.
- Combining retrieved context with an LLM to answer user questions accurately.